# exp142 trajectory-aware PF transition prior train

Train-side pseudo-tail audit for PF-Z candidates whose transition prior uses target-free trajectory signals (`dZ/dMD`, `d2Z/dMD2`, prefix slope). The exp106 strict exp072 PF-Z parity path remains the control; trajectory-aware variants are evaluated as candidate additions, not as an immediate `likpf_mean` replacement.


## 1. Setup and configuration


In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, get_nested, load_config
from trajectory_aware_pf_transition_prior import (
    EXP072_TRAIN_FEATURES,
    OUTPUT_PREFIX,
    find_artifact,
    run_audit,
)

config = load_config()
paths = ExperimentPaths()
print('experiment:', get_nested(config, 'experiment.name'))
print('route:', get_nested(config, 'experiment.route'))
print('status:', get_nested(config, 'experiment.status'))
print('parent:', get_nested(config, 'lineage.parent'))
print('cache parent:', get_nested(config, 'lineage.cache_parent'))
print('strict_pf_z:', json.dumps(get_nested(config, 'model.strict_pf_z'), indent=2))
print('trajectory_pf_z:', json.dumps(get_nested(config, 'model.trajectory_pf_z'), indent=2))
print('artifacts dir:', paths.artifacts_dir)


## 2. Input preview


In [ ]:
cache_path = find_artifact(
    EXP072_TRAIN_FEATURES,
    get_nested(config, 'data.exp072_train_feature_cache_local'),
)
print('exp072 cache:', cache_path)
cache_header = pd.read_csv(cache_path, nrows=0).columns.tolist()
print('cache columns:', len(cache_header))
required = [
    'id', 'well', 'target', 'last_known_tvt', 'md_since',
    'pf_z', 'pf_ancc', 'beam_mean_d', 'likpf_mean_d',
]
missing = [col for col in required if col not in cache_header]
print('missing required columns:', missing)
preview_cols = [col for col in required if col in cache_header]
display(pd.read_csv(cache_path, usecols=preview_cols, nrows=5))


## 3. Variant plan


In [ ]:
variants = get_nested(config, 'model.trajectory_pf_z.transition_variants') or []
variant_table = pd.DataFrame(variants)
display(variant_table)
print('trajectory scales:', get_nested(config, 'model.trajectory_pf_z.scales'))
print('quality metrics:', ['mean_neff_frac', 'min_neff_frac', 'mean_resample_count', 'mean_collapse_rate', 'mean_particle_std'])


## 4. Run PF audit


In [ ]:
summary = run_audit(config)
print(json.dumps(summary, indent=2, sort_keys=True)[:8000])


## 5. Metrics and artifacts


In [ ]:
artifact_dir = paths.artifacts_dir
metrics_path = artifact_dir / f'{OUTPUT_PREFIX}_candidate_metrics.csv'
quality_path = artifact_dir / f'{OUTPUT_PREFIX}_trajectory_pf_z_quality.csv'
bucket_path = artifact_dir / f'{OUTPUT_PREFIX}_bucket_metrics.csv'
by_well_path = artifact_dir / f'{OUTPUT_PREFIX}_by_well.csv'
summary_path = artifact_dir / f'{OUTPUT_PREFIX}_summary.json'
parity_path = artifact_dir / f'{OUTPUT_PREFIX}_parity_diff.csv.gz'
wide_path = artifact_dir / f'{OUTPUT_PREFIX}_candidate_wide.csv.gz'

metrics = pd.read_csv(metrics_path)
display(metrics.head(30))

traj_quality = pd.read_csv(quality_path)
display(traj_quality.groupby('variant').agg({
    'well': 'nunique',
    'mean_neff_frac': 'mean',
    'min_neff_frac': 'min',
    'mean_resample_count': 'mean',
    'mean_collapse_rate': 'mean',
    'mean_particle_std': 'mean',
}).reset_index())

bucket = pd.read_csv(bucket_path)
display(bucket[bucket['bucket_family'].isin(['abs_dzdmd', 'abs_d2zdmd2'])].head(40))
print('artifacts:', sorted(p.name for p in artifact_dir.glob(f'{OUTPUT_PREFIX}*')))
print('summary:', summary_path)
print('parity diff:', parity_path)
print('candidate wide:', wide_path)
